# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
"""
Lane: Refresh / Content Opportunity Scoring.

Task type: Ranking / scoring, backed by a binary classification model underneath.

The decision a reviewer faces isn't "is this page declining, yes or no" 
in isolation,it's "out of everything live, which ~50 pages do I open first this week?" 
That's a ranking question ("which ones first?"), not a plain classification question. 
The classifier's probability output is the engine that produces the score, but the deliverable is a ranked queue, not a label.

"""


In [ ]:
import pandas as pd
import numpy as np

TASK_TYPE = "Ranking / scoring (probability-based), built on a binary classifier"
print("Task type for this lane:", TASK_TYPE)

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
"""

Target: is_declining_label=(trend_direction=="down"),defined in this repo's pipeline.

trend_direction comes from trend_pct, which compares impressions_last_30d against
impressions_prev_30d - a real observed change, not a threshold someone invented after looking at results. 
So this passes the "observed, not defined" test.

The catch: because is_declining_label is derived from trend_direction and trend_pct,
those two columns can never be used as model features later - that would just be
re-deriving the label(leakage), not learning a pattern.

"""

In [ ]:
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Rows:", len(df))
print(df[["trend_direction", "trend_pct", "is_declining_label"]].head())
print("\nLabel balance:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

LEAKY_COLUMNS = ["trend_direction", "trend_pct"]
print("\nExcluded from any future feature set (label-derived):", LEAKY_COLUMNS)

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
"""

Metric: Precision@50.

A reviewer can act on roughly 50 pages a week (fixed capacity). 
Precision@50 asks:of the top 50 pages the model surfaces, what fraction are actually declining? 
That matches the real decision better than accuracy or ROC-AUC, which score the whole ranking equally even though the reviewer only touches the top of it.

Computed below with days_since_last_update — not trend_pct, since that column builds the label and using it would be leakage, not a real baseline.

"""

In [ ]:
def precision_at_k(frame, score_col, label_col, k=50, ascending=False):
    ranked = frame.sort_values(score_col, ascending=ascending).head(k)
    return ranked[label_col].mean()

naive_precision_50 = precision_at_k(df, "days_since_last_update", "is_declining_label",k=50, ascending=False)
random_baseline = df["is_declining_label"].mean()

print(f"Base rate (random pick): {random_baseline:.3f}")
print(f"Naive single-signal Precision@50 (staleness): {naive_precision_50:.3f}")
print("\nFor comparison, outputs/model_report.md in this repo already logs:")
print("  baseline_rules  Precision@50 = 0.240")
print("  random_forest   Precision@50 = 0.740")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
"""

One row = one content item (page), identified by content_id, belonging to one client_id, with metrics aggregated over a trailing 90-day window. 
content_id and client_id are pseudonyms — used only for grouping/dedup, never as features.

Filter applied: impressions_90d > 0, content_age_days >= 90, dedup on content_id(same filter used to scope this lane in ML-02).

"""

In [ ]:
df_lane = df.drop_duplicates(subset="content_id").copy()
df_lane = df_lane[(df_lane["impressions_90d"] > 0) & (df_lane["content_age_days"] >= 90)]

print("Unit of analysis: one row = one content item (page)")
print("Lane slice shape:", df_lane.shape)

display_cols = ["content_id", "client_id", "content_type", "content_age_days","impressions_90d", "clicks_90d", "avg_position", "ctr","trend_direction", "trend_pct", "is_declining_label"]
df_lane[display_cols].head(8)

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
"""

A fixed rule (e.g. "flag anything where trend_pct < -10") only looks at one signal at a time.
Below, several individual signals are checked against the label on their own - each is weak by itself. 
Real decline shows up as a combination of things moving together (falling impressions + demand + staleness + thin engagement),
and that combination isn't fixed - it shifts by content type and client. 
That's why a model that weighs many signals together (random_forest, 0.740) beats a hand-built rule (baseline_rules, 0.240).

"""

In [ ]:
candidate_signals = ["impressions_90d", "avg_position", "ctr", "engagement_rate",
                     "days_since_last_update", "content_age_days"]

corrs = (df_lane[candidate_signals + ["is_declining_label"]]
         .corr(numeric_only=True)["is_declining_label"]
         .drop("is_declining_label")
         .sort_values(key=abs, ascending=False))

print("Correlation of individual signals with is_declining_label:")
print(corrs.round(3))
print("\nEvery signal is weak alone — no clean if-statement cutoff exists on any one column.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.